[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/38_grpo_loss.ipynb)

# 🔴 Hard: GRPO Loss

Implement the **Group Relative Policy Optimization (GRPO)** loss — a group-wise, baseline-subtracted REINFORCE objective commonly used in RLAIF (reinforcement learning from AI feedback).

Given a batch of log-probabilities, scalar rewards, and group ids (one group per prompt), define the within-group normalized advantages:

$$A_i = \frac{r_i - \bar r_{g(i)}}{\text{std}_{g(i)} + \epsilon}$$

where \(\bar r_{g(i)}\) and \(\text{std}_{g(i)}\) are the mean and standard deviation of rewards in the group of example \(i\).

The GRPO loss is then the negative advantage-weighted log-probability:

$$\mathcal{L}_{\text{GRPO}} = -\mathbb{E}_i \big[\,\text{stop\_grad}(A_i)\, \log \pi_\theta(y_i)\big].$$

### Signature
```python
from torch import Tensor

def grpo_loss(logps: Tensor, rewards: Tensor, group_ids: Tensor,
              eps: float = 1e-5) -> Tensor:
    """GRPO loss over a batch.

    logps: (B,) policy log-probs for each sampled response
    rewards: (B,) scalar rewards for each response
    group_ids: (B,) integers, same id = same prompt/group
    returns: scalar loss (Tensor)
    """
```

In [1]:
import torch
import torch.nn.functional as F

In [14]:
# ✏️ YOUR IMPLEMENTATION HERE

from torch import Tensor

def grpo_loss(logps: Tensor, rewards: Tensor, group_ids: Tensor,
              eps: float = 1e-5) -> Tensor:
    unique_ids = group_ids.unique()
    advantages = torch.empty_like(rewards)
    for gid in unique_ids:
        mask = (group_ids == gid)
        rg = rewards[mask]
        rewards_mean = torch.mean(rg)
        rewards_stdev = torch.std(rg,unbiased=False)
        advantages[mask] = (rg - rewards_mean) / (rewards_stdev+eps)

    return -torch.mean(advantages.detach()*logps)

In [15]:
# 🧪 Debug
logps = torch.tensor([0.0, -0.5, -1.0, -1.5])
rewards = torch.tensor([1.0, 0.8, 0.2, 0.0])
group_ids = torch.tensor([0, 0, 1, 1])
print('Loss:', grpo_loss(logps, rewards, group_ids).item())

Loss: -0.24997496604919434


In [16]:
# ✅ SUBMIT
from torch_judge import check
check('grpo_loss')


🧪 Testing: GRPO (Group Relative Policy Optimization) Loss (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Basic shape & type (2.8ms)
  ✅ [2/4] Numeric check vs reference (3.8ms)
  ✅ [3/4] Gradient flows to logps only (3.5ms)
  ✅ [4/4] Group-wise normalization (8.0ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (18.1ms total)
  Progress saved. Run status() to see your dashboard.

